# Stage A (eval) — the ablation-tier evaluation bank

`kaggle_stage_a.ipynb` builds the TRAINING bank. This builds the EVALUATION
bank, which `run_ablation.py --eval-bank` scores the robustness table from,
and which rung **A5** needs for BOTH backbones before it can fuse them.

Three differences from the training notebook, and each is a thing that fails
silently if you get it wrong:

1. **A different manifest.** `eval_manifest.parquet`, which joins the training
   tree with the organisers' demo benchmark. Published separately because
   neither image Dataset carries it.
2. **A different root.** The eval manifest is re-rooted onto the COMMON
   ANCESTOR of the two trees, so its `rel_path` starts `normalized/` or
   `demo/`. The image Datasets mount one level below that, so this notebook
   builds a two-link symlink farm rather than using `unify_mounts` (which
   looks for top-level names that are not there).
3. **No shard by default.** 25,332 rows x 20 conditions = 506,640 forwards,
   which fits one session. Set `N_SHARDS > 1` only if yours will not.

The rel_paths must stay byte-identical to the local ones: `manifest_sha256`
is taken over them in order, and `eval.fusion.assert_fusion_parents` refuses
to fuse two banks whose fingerprints differ. That is why the manifest is
uploaded rather than rebuilt here.

In [ ]:
# ============ THE ONLY LINES YOU NORMALLY EDIT ============
SHARD_INDEX = 0          # leave at 0 unless N_SHARDS > 1
N_SHARDS    = 1          # 1 = the whole bank in this session
SMOKE       = True       # True = prove the chain in minutes; False = the real run
# ==========================================================

BACKBONE = "siglip2l"    # dinov3l's eval bank is the A4500's job (and gated)
TIER     = "ablation"    # sets conditions, splits and row budget together
SEED     = 20260827      # must match scripts/extract_eval_bank.py

BATCH_SIZE       = 64
CHECKPOINT_EVERY = 200

REPO_URL = "https://github.com/bersamin12/robust-aigc-detection"
BRANCH   = "feat/robust-aigc-detection"
REPO_DIR = "/kaggle/working/robust-aigc-detection"

# Attach all three under Add data.
TRAIN_MOUNT    = "/kaggle/input/techjam-aigc-train"
BENCH_MOUNT    = "/kaggle/input/techjam-aigc-benchmark"
MANIFEST_GLOB  = "/kaggle/input/techjam-aigc-eval-manifest/eval_manifest.parquet"

# The symlink farm. /kaggle/temp does not persist, which is correct: it holds
# no bytes, only two links, and is rebuilt in seconds on a resumed session.
EVAL_ROOT = "/kaggle/temp/aigcdet_eval_root"

OUT_DIR = (f"/kaggle/working/banks/eval_{BACKBONE}"
           + (f"_shard{SHARD_INDEX}" if N_SHARDS > 1 else ""))
print("out:", OUT_DIR)

## 1. Get the code

Public repo, shallow clone, and a `reset --hard` on re-run so a resumed session
picks up any fix without a stale working tree. Nothing here is authenticated.

In [ ]:
import glob, importlib, os, subprocess, sys, time

def sh(argv, **kw):
    """Run a command, show it, and fail loudly rather than continuing."""
    print("$", " ".join(str(a) for a in argv))
    return subprocess.run([str(a) for a in argv], check=True, **kw)

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    sh(["git", "-C", REPO_DIR, "fetch", "--depth", "1", "origin", BRANCH])
    sh(["git", "-C", REPO_DIR, "reset", "--hard", f"origin/{BRANCH}"])
else:
    sh(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR])

# Two directories, for two different importers. `notebooks/` is where
# `kaggle_bootstrap` lives. `src/` is where the `aigcdet` package lives, and it
# is put on the path HERE rather than left to the `pip install -e` in the next
# section, because an editable install registers itself through a .pth file
# that site.py reads at INTERPRETER START. A kernel that was already running
# when pip finished never sees it. Every extraction is a subprocess with a
# fresh interpreter, so those work either way -- but an in-kernel
# `from aigcdet...` raises ModuleNotFoundError, several cells later, with the
# install cell reporting success.
for _sub in ("notebooks", "src"):
    _p = os.path.join(REPO_DIR, _sub)
    if _p not in sys.path:
        sys.path.insert(0, _p)
importlib.invalidate_caches()
import kaggle_bootstrap as kb
importlib.reload(kb)

sh(["git", "-C", REPO_DIR, "log", "--oneline", "-1"])
print("helper loaded from", kb.__file__)

## 2. Install — without losing Kaggle's torch

This is the step that ends sessions. `pip install -e .` hands pip the
`torch>=2.0` line from `pyproject.toml` and invites it to resolve a torch built
for a different CUDA than this machine's drivers; you get a `torch` that cannot
see the GPU and no way back except a factory reset.

So: the project goes in with `--no-deps` (a pure path registration, which is all
it is needed for), everything else is installed **only if genuinely missing**,
and nothing CUDA-matched is touched at all. `transformers` is the one exception
— Kaggle images move and the project needs ≥4.53 for DINOv3 — and it is
upgraded with `--no-deps` too.

**Print the plan before running it.** If you ever see `torch` in that list,
stop.

In [ ]:
def installed_version(dist):
    try:
        import importlib.metadata as im
        return im.version(dist)
    except Exception:
        return None

plan = kb.install_plan(os.path.join(REPO_DIR, "pyproject.toml"), REPO_DIR,
                       transformers_version=installed_version("transformers"))

print("pip plan:")
for cmd in plan:
    print("   ", " ".join(cmd))

assert not any(w.split("=")[0].split(">")[0] in ("torch", "torchvision", "triton")
               for cmd in plan for w in cmd), "STOP: the plan would touch torch"

for cmd in plan:
    sh(cmd, capture_output=True, text=True)
print("\ninstall done")

# Prove the project is importable IN THIS KERNEL, here, where the remedy is
# still "re-run the two cells above". Without this the first in-kernel
# `from aigcdet...` is in the auth section, and a ModuleNotFoundError there
# reads as an auth problem rather than a path one.
importlib.invalidate_caches()
from aigcdet.features.backbones import BACKBONES as _B
print("aigcdet importable:", len(_B), "backbones registered")

### 2b. Check the environment before paying for anything

Every problem this cell can report makes the 1.2 GB model download pointless,
so it runs before the download rather than after it.

If it tells you `transformers` was upgraded: **restart the kernel**
(Run → Restart session) and re-run from cell 0. A module already imported at
the old version stays imported.

In [ ]:
import platform

torch_v = installed_version("torch")
tf_v    = installed_version("transformers")
problems = kb.environment_problems(platform.python_version(), torch_v, tf_v)

print(f"python {platform.python_version()}  torch {torch_v}  transformers {tf_v}")
if problems:
    for p in problems:
        print("\nPROBLEM:", p)
    raise SystemExit("fix the above before continuing")

import torch
print("cuda available:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")
assert torch.cuda.is_available(), (
    "no GPU. Settings > Accelerator > GPU, then restart the session. "
    "Extraction on CPU will not finish inside a session.")

## 3. HuggingFace auth

**Only if your `BACKBONE` is gated.** SigLIP2 (Apache-2.0) and CLIP (MIT) are
public: the cell below will say so and move on, and you need no token at all.
DINOv3 is gated behind Meta's licence, and then two separate things must be
true — from this notebook they fail identically, as a 401/403 on
`from_pretrained`:

1. **Your own** HuggingFace account has accepted the licence at the model page.
   Acceptance is per account — the project owner's acceptance does nothing for
   yours.
2. A read token from that same account is attached to this notebook as a Kaggle
   Secret named `HF_TOKEN`.

**Never paste a token into a cell.** This repo is public and a notebook is
committed with its cell source. Add-ons → Secrets is the whole reason that
mechanism exists.

In [ ]:
# Both read off the registry, so they follow BACKBONE rather than being
# typed again here. DINOv3 is gated behind Meta's licence; SigLIP2
# (Apache-2.0) and CLIP (MIT) are not, and the fleet should not be
# stopped for a token its run never uses.
from aigcdet.features.backbones import BACKBONES
MODEL_ID = BACKBONES[BACKBONE].hf_id
GATED    = kb.requires_hf_token(BACKBONE)

try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
except Exception:
    secrets = None

token = kb.hf_token(secrets)
for line in kb.hf_auth_advice(token, MODEL_ID, gated=GATED):
    print(line)

if token:
    # Exported for `transformers` to pick up. Never printed, never written to a
    # file that leaves this session.
    os.environ["HF_TOKEN"] = token
    os.environ["HUGGING_FACE_HUB_TOKEN"] = token
elif GATED:
    raise SystemExit("no HuggingFace token -- see the instructions above")

## The symlink farm, and the manifest it has to satisfy

`unify_mounts` locates a manifest's root by looking for its top-level names
inside each mount. The eval manifest's names are `normalized/` and `demo/`,
and the Datasets mount their CONTENTS at that level instead -- so the farm is
built explicitly here. Two links, checked against the manifest rather than
assumed.

In [ ]:
import pandas as pd, shutil

MANIFEST = sorted(glob.glob(MANIFEST_GLOB))
assert MANIFEST, f"no eval manifest attached (looked for {MANIFEST_GLOB})"
MANIFEST = MANIFEST[0]
for m in (TRAIN_MOUNT, BENCH_MOUNT):
    assert os.path.isdir(m), f"not attached: {m}"

rel = pd.read_parquet(MANIFEST, columns=["rel_path"])["rel_path"]
EXPECTED = sorted({p.split("/")[0] for p in rel})
print("manifest expects top-level:", EXPECTED)

# normalized/ -> the training tree; demo/ -> the organisers' benchmark.
LINKS = {"normalized": TRAIN_MOUNT, "demo": BENCH_MOUNT}
assert set(LINKS) == set(EXPECTED), (
    f"manifest wants {EXPECTED}, this notebook links {sorted(LINKS)}. "
    "Fix the mapping rather than the manifest -- rel_path is the fingerprint.")

shutil.rmtree(EVAL_ROOT, ignore_errors=True)
os.makedirs(EVAL_ROOT, exist_ok=True)
for name, target in LINKS.items():
    os.symlink(target, os.path.join(EVAL_ROOT, name))
print("root:", EVAL_ROOT, "->", sorted(os.listdir(EVAL_ROOT)))

# Prove the links actually resolve to images, on real rows, before an hour of
# GPU discovers otherwise. A mount attached under a different slug produces a
# farm that lists correctly and resolves to nothing.
missing = [p for p in rel.sample(200, random_state=SEED)
           if not os.path.exists(os.path.join(EVAL_ROOT, p))]
assert not missing, f"{len(missing)} of 200 sampled rows do not resolve, e.g. {missing[:3]}"
print("200 sampled rows all resolve")

## Plan, smoke, run

`--dry-run` prints the condition axis, the split budget and the forward count
without loading a backbone, so the tier is visible before anything is paid
for.

In [ ]:
EXTRACT = os.path.join(REPO_DIR, "scripts", "extract_eval_bank.py")

def eval_argv(*, dry=False, limit=None, out=OUT_DIR):
    a = [sys.executable, EXTRACT,
         "--manifest", MANIFEST, "--backbone", BACKBONE,
         "--out", out, "--tier", TIER, "--root", EVAL_ROOT,
         "--batch-size", str(BATCH_SIZE), "--seed", str(SEED),
         "--checkpoint-every", str(CHECKPOINT_EVERY), "--resume"]
    if N_SHARDS > 1:
        a += ["--shard", f"{SHARD_INDEX}/{N_SHARDS}"]
    if dry:
        a += ["--dry-run"]
    if limit is not None:
        a += ["--limit", str(limit)]
    return a

kb.run_streaming(eval_argv(dry=True))

In [ ]:
t0 = time.time()
if SMOKE:
    rc = kb.run_streaming(eval_argv(limit=40, out="/kaggle/working/banks/_smoke"))
    assert rc == 0, "smoke run failed -- fix it here, not 90 minutes in"
    dt = time.time() - t0
    print(f"\n40 images in {dt:.0f}s")
    print(f"estimated full run: {dt/40*25332/3600:.1f} h "
          f"(a Kaggle GPU session is 9 h)")
    print("\nSet SMOKE = False, re-run cells 0-5, then run the cell below.")
else:
    print("SMOKE is False -- skip to the extraction cell.")

In [ ]:
assert not SMOKE, "set SMOKE = False first"
rc = kb.run_streaming(eval_argv())
print("exit", rc)

## Verify, then publish

`check_invariants` streams `feats.npy` for non-finite values FIRST. That check
exists because a 5-hour DINOv3 bank on 2026-08-29 was 131,116 rows of NaN and
passed every other post-condition, including its row count.

In [ ]:
from aigcdet.features.bank import FeatureBank

bank = FeatureBank(OUT_DIR)
bank.check_invariants()
n_expected = len(pd.read_parquet(MANIFEST))
print("config:", bank.config)
print(f"rows {len(bank.meta)}")
print("COMPLETE" if len(bank.meta) == bank.config["n_images"]
      else f"INCOMPLETE -- {bank.config['n_images'] - len(bank.meta)} rows remain; "
           "re-run the extraction cell, it resumes")
print("\nSave Version > Quick Save, then Output > New Dataset named:")
print("   ", os.path.basename(OUT_DIR).replace("_", "-"))